<a href="https://colab.research.google.com/github/ashut0sh16/Object-Counting/blob/main/YouTube_RAG_with_Huggingface_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install youtube-transcript-api langchain-community faiss-cpu sentence-transformers transformers

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# Step 1a - indexing (Document ingestion)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving transcript.txt to transcript (1).txt


In [ ]:
with open("transcript.txt", "r", encoding="utf-8") as f:
    transcript = f.read()

print(transcript[:500])  # just to verify

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful 


In [ ]:
from langchain_core.documents import Document

doc = Document(page_content=transcript)

# Step 1b - Indexing (Text Splitting)

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\[.*?\]", "", text)  # remove [music], etc.
    text = re.sub(r"\s+", " ", text)     # remove extra spaces
    text = re.sub(r"(uh|um|you know|like)", "", text)  # remove fillers
    return text.strip()

transcript = clean_text(transcript)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents([doc])

print(len(chunks))

335


In [ ]:
chunks[0]

Document(metadata={}, page_content='the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful')

# Step 1c & 1d (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vector_store.index_to_docstore_id

{0: '56e2b454-5d66-4dad-840b-fd6d8312eb6a',
 1: 'ce656e1d-975c-4de7-9581-825e2d86855f',
 2: '49a089ec-d318-4037-b793-b016f0b7f903',
 3: '9c60a4fe-fd5b-4b6b-bef8-c10dfa63f051',
 4: 'd6816829-c00e-4fa9-824e-86ff73b7b2d1',
 5: 'de252a1e-169f-4ff5-83a8-2e9d15a610f0',
 6: 'df07e07b-147d-4477-a041-3f7c56cc4e16',
 7: '0be172e3-06f0-4d7b-8db2-ac3c3b116ab5',
 8: 'e06b32b4-bd73-4548-8635-b12a0cd97136',
 9: 'cf97f8db-6e70-4f5a-b671-a0094f5a837c',
 10: 'b45161a2-272b-44af-a3d0-0ce966512c01',
 11: '8cc85f39-1394-4504-8a88-63149667f22a',
 12: 'b30302d2-ab1b-4c3b-9056-41942818d916',
 13: 'ae19a7f0-cb88-4769-9ef6-21d4e1f52a55',
 14: '00686213-5f9a-4220-b67e-d8d76a159a63',
 15: '9304981a-416e-4743-97fc-ddca7b46e7e4',
 16: 'a130c941-420e-43ca-9efc-ad8c431ba68e',
 17: '36c0c0fb-cbcf-408d-9c91-efd604d4c9b3',
 18: '3b403287-3e5f-4f97-88d9-6d0f5f91fb28',
 19: '7ab12154-36db-464e-95f3-507cefd64f70',
 20: 'b5d33ad0-5622-413b-8fa4-cfb85b20cb62',
 21: '0e817e15-d19d-4993-b1eb-b46ff5441b8d',
 22: 'c8896ae4-0d7a-

In [ ]:
# vector_store.get_by_ids(['provided_id']) # provide id from above

# Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x78cc334b3c20>, search_kwargs={'k': 5})

In [ ]:
retriever.invoke('What is deepmind')

[Document(id='3a031b57-d4c3-4efe-9246-7a331b35d637', metadata={}, page_content="tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware"),
 Document(id='e6bb0306-1aaa-4b46-a6cd-8f29c04790a0', metadata={}, page_content='that i think we innovated with at deepmind to encourage invention and and uh and innovation was the multi-disciplinary organization we built and we still have today so deepmind originally was a confluence of the of the most cutting-edge knowledge in neuroscience with machine learning engineering and mathematics right and and gaming and th

# Step 3 - Augumentation

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-large",
    max_length=512,
    temperature=0.3
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are an intelligent assistant.

Your job is to understand the context and give a clear, meaningful answer.

- Do NOT repeat the context.
- Summarize and explain in simple words.
- Ignore filler or broken sentences.

Context:
{context}

Question: {question}

Answer:
""",
    input_variables=["context", "question"]
)

In [ ]:
question = "ask the question here"
retrieved_docs = retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='9117565c-e28b-4a29-9351-5fd80c3d07a8', metadata={}, page_content="you only get to ask one question if you do what question would you ask her i would probably ask um what is the true nature of reality i think that's the question i don't know if i'd understand the answer because maybe it would be 42 or something like that but um that's the question i would ask and then there'll be a deep sigh from the systems like all right how do i explain to the excuse me exactly all right let me i don't have time to explain uh maybe i'll draw you a picture that it is i mean"),
 Document(id='9ccca42a-4722-41c5-9593-feb2131538a1', metadata={}, page_content="by the possibility of learning more it's one heck of a one heck of a puzzle we got going on here um so like i mentioned of all the people in the world you're very likely to be the one who creates the agi system um that achieves human level intelligence and goes beyond it so if you got a chance and very well you could be the person that 

In [ ]:
# now we join all those 4 strings into a single string

context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"you only get to ask one question if you do what question would you ask her i would probably ask um what is the true nature of reality i think that's the question i don't know if i'd understand the answer because maybe it would be 42 or something like that but um that's the question i would ask and then there'll be a deep sigh from the systems like all right how do i explain to the excuse me exactly all right let me i don't have time to explain uh maybe i'll draw you a picture that it is i mean\n\nby the possibility of learning more it's one heck of a one heck of a puzzle we got going on here um so like i mentioned of all the people in the world you're very likely to be the one who creates the agi system um that achieves human level intelligence and goes beyond it so if you got a chance and very well you could be the person that goes into the room with the system and have a conversation maybe you only get to ask one question if you do what question would you ask her i would probably as

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text='\nYou are a helpful assistant.\n\nAnswer the question ONLY using the context below.\nIf the answer is not in the context, say "I don\'t know".\n\nExplain clearly and in simple words.\n\nContext:\nyou only get to ask one question if you do what question would you ask her i would probably ask um what is the true nature of reality i think that\'s the question i don\'t know if i\'d understand the answer because maybe it would be 42 or something like that but um that\'s the question i would ask and then there\'ll be a deep sigh from the systems like all right how do i explain to the excuse me exactly all right let me i don\'t have time to explain uh maybe i\'ll draw you a picture that it is i mean\n\nby the possibility of learning more it\'s one heck of a one heck of a puzzle we got going on here um so like i mentioned of all the people in the world you\'re very likely to be the one who creates the agi system um that achieves human level intelligence and goes beyond 

# Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer)

Token indices sequence length is longer than the specified maximum sequence length for this model (641 > 512). Running this sequence through the model will result in indexing errors



You are a helpful assistant.

Answer the question ONLY using the context below.
If the answer is not in the context, say "I don't know".

Explain clearly and in simple words.

Context:
you only get to ask one question if you do what question would you ask her i would probably ask um what is the true nature of reality i think that's the question i don't know if i'd understand the answer because maybe it would be 42 or something like that but um that's the question i would ask and then there'll be a deep sigh from the systems like all right how do i explain to the excuse me exactly all right let me i don't have time to explain uh maybe i'll draw you a picture that it is i mean

by the possibility of learning more it's one heck of a one heck of a puzzle we got going on here um so like i mentioned of all the people in the world you're very likely to be the one who creates the agi system um that achieves human level intelligence and goes beyond it so if you got a chance and very well you c

# Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    "context": retriever| RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

{'context': "our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough to interview you well i'll be impressed if if you were i'd be impressed by myself if you were i don't think we're quite up to that yet but uh maybe you're from the future lex if you did would you tell me is that is that a good thing to tell a language model that's tasked with interviewing that it is in\n\nfor a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain  | prompt | llm | parser

In [ ]:
main_chain.invoke("Summarize the main topic of this video in 3-4 sentences")

'\nYou are a helpful assistant.\n\nAnswer the question ONLY using the context below.\nIf the answer is not in the context, say "I don\'t know".\n\nExplain clearly and in simple words.\n\nContext:\ni think first of all um there\'s there\'s two different things there\'s like what do we understand today yeah what could the human mind understand and what is the totality of what is there to be understood yeah right and so there\'s three consensus you know you can think of them as three larger and larger trees or exploring more branches of that tree and i i think with ai we\'re going to explore that whole lot now the question is is uh you know if you think about what is the totality of what could\n\nthe special human beings in this giant puzzle of ours and it\'s a huge honor that you would take a pause from the bigger puzzle to solve this small puzzle of a conversation with me today it\'s truly an honor and a pleasure thank you thank you i really enjoyed it thanks lex thanks for listening to